# Use a custom function in an AI agent

## Create an Azure AI Foundry project

In [ ]:
AZURE_SUBSCRIPTION_ID=your-subscription-id
RESOURCE_GROUP=rg-agent-function-lab3
LOCATION=eastus
FOUNDRY_RESOURCE_NAME=rs-agent-function
FOUNDRY_PROJECT_NAME=pj-agent-function
MODEL_DEPLOYMENT_NAME=gpt41-deployment
FOUNDRY_PROJECT_ENDPOINT="your_project_endpoint"

Load environment variables

In [1]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.resource import ResourceManagementClient
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()
azure_subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group_name = os.getenv("RESOURCE_GROUP")
location = os.getenv("LOCATION")

Create resource group

In [2]:
# Authenticate
credential = DefaultAzureCredential()
# Create resource group
resource_client = ResourceManagementClient(credential, azure_subscription_id)
print(f"Creating resource group {resource_group_name}...")
resource_client.resource_groups.create_or_update(
    resource_group_name,
    {"location": location}
)
print(f"Resource group {resource_group_name} created successfully.")

Creating resource group rg-agent-function-lab3...
Resource group rg-agent-function-lab3 created successfully.


Create the client connection and variables

In [3]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

foundry_resource_name = os.getenv("FOUNDRY_RESOURCE_NAME")
foundry_project_name = os.getenv("FOUNDRY_PROJECT_NAME")

client = CognitiveServicesManagementClient(
    credential=credential, 
    subscription_id=azure_subscription_id,
    api_version="2025-04-01-preview"
)

Create a Azure AI Foundry project

In [4]:
# Create resource
resource = client.accounts.begin_create(
    resource_group_name=resource_group_name,
    account_name=foundry_resource_name,
    account={
        "location": location,
        "kind": "AIServices",
        "sku": {"name": "S0",},
        "identity": {"type": "SystemAssigned"},
        "properties": {
            "allowProjectManagement": True,
            "customSubDomainName": foundry_resource_name
        }
    }
).result()

# Create default project
project = client.projects.begin_create(
    resource_group_name=resource_group_name,
    account_name=foundry_resource_name,
    project_name=foundry_project_name,
    project={
        "location": location,
        "identity": {
            "type": "SystemAssigned"
        },
        "properties": {}
    }
).result()

### Model Deployment

List avalible model

In [ ]:
az cognitiveservices account list-models \
  -n rs-agent-function \
  -g rg-agent-function-lab3 | \
  jq '.[] | { 
    name: .name, 
    format: .format, 
    version: .version, 
    sku: .skus[0].name, 
    capacity: .skus[0].capacity.default 
  }'

In [ ]:
az cognitiveservices account list-models \
  --name rs-agent-function \
  --resource-group rg-agent-function-lab3 | \
  jq '.[] | select(.name | test("gpt-5"; "i")) | { 
    name: .name, 
    format: .format, 
    version: .version, 
    sku: .skus[0].name, 
    capacity: .skus[0].capacity.default,
    model_category: (
      if (.capabilities | type == "array") then
        if (.capabilities | contains(["ChatCompletion"])) then "Chat"
        elif (.capabilities | contains(["Embedding"])) then "Embeddings" 
        elif (.capabilities | contains(["Completion"])) then "Text Completion"
        else (.capabilities | join(", "))
        end
      else .capabilities
      end
    )
  }'

In [ ]:
{
  "name": "gpt-4.1",
  "format": "OpenAI",
  "version": "2025-04-14",
  "sku": "GlobalStandard",
  "capacity": 10,
  "model_category": {
    "FineTuneTokensMaxValue": "2000000000",
    "FineTuneTokensMaxValuePerExample": "65536",
    "area": "US",
    "assistants": "true",
    "chatCompletion": "true",
    "globalFineTune": "true",
    "responses": "true"
  }
}


Deploy the `gpt-4.1` (CLI Azure) model to the Azure AI Foundry resource.
* This command uses the Foundry resource directly
* `sku-capacity` can be adjusted according to the lab (e.g. 50 for 50K TPM)

In [ ]:
az cognitiveservices account deployment create \
  --name rs-agent-function \
  --resource-group rg-agent-function-lab3 \
  --deployment-name gpt41-deployment \
  --model-name gpt-4.1 \
  --model-version "2025-04-14" \
  --model-format OpenAI \
  --sku-name GlobalStandard \
  --sku-capacity 50

### Get api key and endpoint

SDK Python

In [ ]:
credential = DefaultAzureCredential()
client = CognitiveServicesManagementClient(credential, azure_subscription_id)

# Obtener el recurso AI Foundry
resource_getkey = client.accounts.get(
    resource_group_name=resource_group_name,
    account_name=foundry_resource_name
)

# Endpoint
print("Endpoint:", resource_getkey.properties.endpoint)
project_endpoint = f"{resource_getkey.properties.endpoint}/api/projects/{foundry_project_name}"

Endpoint: https://rs-build-agent.cognitiveservices.azure.com/


In [ ]:
print(project_endpoint)

https://rs-build-agent.cognitiveservices.azure.com//api/projects/pj-build-agent


In [ ]:
# API Key
keys = client.accounts.list_keys(
    resource_group_name=resource_group_name,
    account_name=foundry_resource_name
)
print("API Key:", keys.key1)


CLI Azure

In [ ]:
az cognitiveservices account show \
  --name rs-agent-function \
  --resource-group rg-agent-function-lab3 \
  --query properties.endpoint \
  --output tsv

In [ ]:
az cognitiveservices account keys list \
  --name rs-agent-function \
  --resource-group rg-agent-function-lab3 \
  --query key1 \
  --output tsv